<a href="https://colab.research.google.com/github/th2ch-g/ColabFold_restr/blob/rgi-integration/Boltz1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This is a work-in-progress notebook for [Boltz](https://github.com/jwohlwend/boltz)

⚠️ **Warning to Users:**
- **Alpha Version:** This notebook is currently under active development and is considered a beta version.
- **Usage at Your Own Risk:** Use this notebook at your own discretion and risk.

In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import random
import requests
from string import ascii_uppercase

# Function to add a hash to the jobname
def add_hash(x, y):
    return x + "_" + hashlib.sha1(y.encode()).hexdigest()[:5]

# User inputs
query_sequence = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASKK'  #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
ligand_input = 'N[C@@H](Cc1ccc(O)cc1)C(=O)O'  #@param {type:"string"}
#@markdown  - Use `:` to specify multiple ligands as smile strings
ligand_input_ccd = 'SAH'  #@param {type:"string"}
#@markdown - Use `:` to specify multiple ligands as CCD codes (three-letter codes)
ligand_input_common_name = ''  #@param {type:"string"}
#@markdown - Use `:` to specify multiple ligands with their common name (e.g. Aspirin; SMILES fetched from [PubChem](https://pubchem.ncbi.nlm.nih.gov) API)
dna_input = ''  #@param {type:"string"}
#@markdown - Use `:` to specify multiple DNA sequences
jobname = 'test'  #@param {type:"string"}

use_rgi = False #@param {type:"boolean"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
import json
for _k, _v in json.loads(os.environ.get('BOLTZ_NB_OVERRIDES', '{}')).items():
    if _k in globals(): globals()[_k] = _v

# Clean up the query sequence and jobname
query_sequence = "".join(query_sequence.split())
ligand_input = "".join(ligand_input.split())
ligand_input_ccd = "".join(ligand_input_ccd.split())
ligand_input_common_name = "".join(ligand_input_common_name.split())
dna_input = "".join(dna_input.split())
basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# Check if a directory with jobname exists
def check(folder):
    return not os.path.exists(folder)

if not check(jobname):
    n = 0
    while not check(f"{jobname}_{n}"):
        n += 1
    jobname = f"{jobname}_{n}"

# Make directory to save results
os.makedirs(jobname, exist_ok=True)

from string import ascii_uppercase

# Split sequences on chain breaks
protein_sequences = query_sequence.strip().split(':') if query_sequence.strip() else []
ligand_sequences = ligand_input.strip().split(':') if ligand_input.strip() else []
ligand_sequences_ccd = ligand_input_ccd.strip().split(':') if ligand_input_ccd.strip() else []
ligand_sequences_common_name = ligand_input_common_name.strip().split(':') if ligand_input_common_name.strip() else []
dna_sequences = dna_input.strip().split(':') if dna_input.strip() else []

def get_smiles(compound_name):
    autocomplete_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/autocomplete/compound/{compound_name}/json?limit=1"
    autocomplete_response = requests.get(autocomplete_url)
    if autocomplete_response.status_code != 200:
        return None

    autocomplete_data = autocomplete_response.json()
    if autocomplete_data.get("status", {}).get("code") != 0 or autocomplete_data.get("total", 0) == 0:
        return None

    suggested_compound = autocomplete_data.get("dictionary_terms", {}).get("compound", [])
    if not suggested_compound:
        return None
    suggested_compound_name = suggested_compound[0]

    smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{suggested_compound_name}/property/CanonicalSMILES/JSON"
    smiles_response = requests.get(smiles_url)
    if smiles_response.status_code != 200:
        return None

    smiles_data = smiles_response.json()
    properties = smiles_data.get("PropertyTable", {}).get("Properties", [])
    if len(properties) == 0:
        return None

    return properties[0].get("CanonicalSMILES")

smiles_cache = {}
for name in ligand_sequences_common_name:
    if name not in smiles_cache:
        smiles_cache[name] = get_smiles(name)
        if smiles_cache[name] is not None:
          print(f"Mapped compound {name} to {smiles_cache[name]}")

    if smiles_cache[name] is not None:
        ligand_sequences.append(smiles_cache[name])

# Initialize chain labels starting from 'A'
chain_labels = iter(ascii_uppercase)

fasta_entries = []
csv_entries = []
chain_label_to_seq_id = {}
seq_to_seq_id = {}
seq_id_counter = 0  # Counter for unique sequences

# Process protein sequences
for seq in protein_sequences:
    seq = seq.strip()
    if not seq:
        continue  # Skip empty sequences
    chain_label = next(chain_labels)
    # Check if sequence has been seen before
    if seq in seq_to_seq_id:
        seq_id = seq_to_seq_id[seq]
    else:
        seq_id = f"{jobname}_{seq_id_counter}"
        seq_to_seq_id[seq] = seq_id
        seq_id_counter += 1
        # For CSV file (for ColabFold), add only unique sequences
        csv_entries.append((seq_id, seq))
    chain_label_to_seq_id[chain_label] = seq_id
    # For FASTA file
    msa_path = os.path.join(jobname, f"{seq_id}.a3m")
    header = f">{chain_label}|protein|{msa_path}"
    sequence = seq
    fasta_entries.append((header, sequence))

# Process ligand sequences (assumed to be SMILES strings)
for lig in ligand_sequences:
    lig = lig.strip()
    if not lig:
        continue  # Skip empty ligands
    chain_label = next(chain_labels)
    lig_type = 'smiles'
    header = f">{chain_label}|{lig_type}"
    sequence = lig
    fasta_entries.append((header, sequence))

# Process DNA sequences (NO MSA is generated)
for seq in dna_sequences:
    seq = seq.strip()
    if not seq:
        continue  # Skip empty sequences
    chain_label = next(chain_labels)
    lig_type = 'DNA'
    header = f">{chain_label}|{lig_type}"
    sequence = seq
    fasta_entries.append((header, sequence))

# Process ligand sequences (CCD codes)
for lig in ligand_sequences_ccd:
    lig = lig.strip()
    if not lig:
        continue  # Skip empty ligands
    chain_label = next(chain_labels)
    lig_type = 'ccd'
    header = f">{chain_label}|{lig_type}"
    sequence = lig.upper()  # Ensure CCD codes are uppercase
    fasta_entries.append((header, sequence))

# Write the CSV file for ColabFold
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
    text_file.write("id,sequence\n")
    for seq_id, seq in csv_entries:
        text_file.write(f"{seq_id},{seq}\n")

# Write the FASTA file
queries_fasta = os.path.join(jobname, f"{jobname}.fasta")
with open(queries_fasta, 'w') as f:
    for header, sequence in fasta_entries:
        f.write(f"{header}\n{sequence}\n")

# Optionally, print the output for verification
#print(f"Generated FASTA file '{queries_fasta}':\n")
#for header, sequence in fasta_entries:
#    print(f"{header}\n{sequence}\n")

print('Chain IDs for restraints:')
for header, sequence in fasta_entries:
    print(header.split('|')[0].lstrip('>'), header.split('|')[1], len(sequence) if header.split('|')[1] == 'protein' else '')


In [ ]:
#@title Install dependencies
import shutil, subprocess, sys
from pathlib import Path
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
_colabfold_source = os.environ.get('RGI_COLABFOLD_SOURCE', 'git+https://github.com/th2ch-g/ColabFold_restr.git@rgi-integration')
_toolkit_source = os.environ.get('RGI_TOOLKIT_SOURCE', 'git+https://github.com/cddlab/rgi_toolkit.git@9f4f2068035074c5e2a9ae506f386ef26a05d3ee')
subprocess.run(['uv', 'pip', 'install', '--python', sys.executable, _colabfold_source, _toolkit_source], check=True)
# Boltz requires Python 3.12; isolate it from Colab's Python and NumPy versions.
boltz_env = Path('.cache') / ('boltz-rgi' if use_rgi else 'boltz-vanilla')
if not (boltz_env / 'bin/python').exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', str(boltz_env)], check=True)
boltz_python = str(boltz_env / 'bin/python')
_boltz_source = ('git+https://github.com/cddlab/boltz_restr.git@rgi-integration' if use_rgi else 'boltz==2.2.1')
subprocess.run(['uv', 'pip', 'install', '--python', boltz_python, _boltz_source], check=True)
if use_rgi:
    subprocess.run(['uv', 'pip', 'install', '--python', boltz_python, _toolkit_source], check=True)


In [ ]:
#@title Generate MSA with ColabFold
if msa_mode == 'mmseqs2_server' and csv_entries:
    from colabfold.colabfold import run_mmseqs2
    for seq_id, seq in csv_entries:
        msa_path = Path(jobname) / f'{seq_id}.a3m'
        if not msa_path.exists():
            a3m = run_mmseqs2([seq], str(Path(jobname) / (seq_id + '_msa')), use_env=True)[0]
            msa_path.write_text(a3m)
else:
    print('Single sequence: skipping the MSA server.')


In [ ]:
#@title RGI restraints (optional)
#@markdown First set **use_rgi** in the input cell. Run this cell again after changing molecules.
#@markdown **distance** constrains the distance between the centroids of two groups (Angstrom).
#@markdown Residues are numbered from 1 within each chain. Example: `5-20,31`.
#@markdown **ligand_geometry** preserves ligand geometry and limits clashes; add a ligand first.
#@markdown **custom** accepts any RGI-toolkit restraint, including RMSD, angles, base pairs and custom energies.
rgi_preset = "distance" #@param ["distance", "ligand_geometry", "distance+ligand_geometry", "custom"]
rgi_chain1 = "A" #@param {type:"string"}
rgi_residues1 = "1-10" #@param {type:"string"}
rgi_chain2 = "A" #@param {type:"string"}
rgi_residues2 = "40-50" #@param {type:"string"}
rgi_atoms = "all" #@param ["all", "CA", "backbone"]
rgi_distance = 25.0 #@param {type:"number"}
rgi_tolerance = 0.0 #@param {type:"number"}
#@markdown Tolerance 0 targets one distance; a positive tolerance permits distance +/- tolerance.
rgi_conformer_chains = "ligands" #@param {type:"string"}
#@markdown `ligands` selects all ligand chains. Alternatively enter chain IDs such as `B,C`.
#@markdown Polymer geometry is advanced: select its chains explicitly and consider `monomer_library: true`.
rgi_custom = "" #@param {type:"string"}
rgi_config_path = "" #@param {type:"string"}
#@markdown For custom mode, paste YAML/JSON or upload a file using Colab's Files panel and enter its path.
#@markdown Upload referenced structures too. A config file's reference paths are relative to that file.

for _k, _v in json.loads(os.environ.get('BOLTZ_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v

rgi_config = None
if use_rgi:
  from rgi_toolkit.notebook import make_config, residue_selection
  _selection1 = residue_selection(rgi_chain1, rgi_residues1, rgi_atoms) if 'distance' in rgi_preset else ''
  _selection2 = residue_selection(rgi_chain2, rgi_residues2, rgi_atoms) if 'distance' in rgi_preset else ''
  rgi_config = make_config(rgi_preset, selection1=_selection1, selection2=_selection2,
      distance=rgi_distance, tolerance=rgi_tolerance, custom=rgi_custom, config_path=rgi_config_path)
  print(json.dumps(rgi_config, indent=2))
else:
  print('Vanilla prediction: RGI is OFF.')

from colabfold.rgi.boltz import notebook_input
boltz_input = notebook_input(fasta_entries, config=rgi_config,
    conformer_chains=rgi_conformer_chains, single_sequence=(msa_mode == 'single_sequence'))


In [ ]:
#@title Predict structure using Boltz-1
import yaml
num_recycles = 3 #@param {type:"integer"}
num_diffusion_samples = 1 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
# Use YAML so the predictor can read the shared RGI config and entity opt-ins.
boltz_input = notebook_input(fasta_entries, config=rgi_config if use_rgi else None,
    conformer_chains=rgi_conformer_chains, single_sequence=(msa_mode == 'single_sequence'))
input_yaml = Path(jobname) / f'{jobname}.yaml'
input_yaml.write_text(yaml.safe_dump(boltz_input, sort_keys=False))
subprocess.run([boltz_python, '-m', 'boltz.main', 'predict', str(input_yaml),
    '--model', 'boltz1', '--out_dir', jobname, '--cache', str(Path('.cache') / 'boltz-weights'),
    '--recycling_steps', str(num_recycles), '--diffusion_samples', str(num_diffusion_samples),
    '--seed', str(seed), '--override'], check=True)
import glob
if not glob.glob(f'{jobname}/boltz_results_{jobname}/predictions/{jobname}/*.cif'):
    raise RuntimeError('Prediction produced no structure files. Check the log above.')
print('RGI ON: check nonzero built-spec counts and final energies above.' if use_rgi else 'Vanilla prediction complete.')


In [ ]:
#@title Download results
# Import necessary modules
import os
import zipfile
from google.colab import files
import glob

# Ensure 'jobname' variable is defined
# jobname = 'test_abcde'  # Uncomment and set if not already defined

# Name of the zip file
zip_filename = f"results_{jobname}.zip"

# Create a zip file and add the specified files without preserving directory structure
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    coverage_png_files = glob.glob(os.path.join(jobname, '*_coverage.png'))
    a3m_files = glob.glob(os.path.join(jobname, '*.a3m'))
    for file in coverage_png_files + a3m_files + [str(input_yaml)]:
        arcname = os.path.basename(file)  # Use only the file name
        zipf.write(file, arcname=arcname)

    cif_files = glob.glob(os.path.join(jobname, f'boltz_results_{jobname}', 'predictions', jobname, '*.cif'))
    for file in cif_files:
        arcname = os.path.basename(file)  # Use only the file name
        zipf.write(file, arcname=arcname)

    hparams_file = os.path.join(jobname, f'boltz_results_{jobname}', 'lightning_logs', 'version_0', 'hparams.yaml')
    if os.path.exists(hparams_file):
        arcname = os.path.basename(hparams_file)  # Use only the file name
        zipf.write(hparams_file, arcname=arcname)
    else:
        print(f"Warning: {hparams_file} not found.")

# Download the zip file
files.download(zip_filename)
